In [12]:
# Celda 1: Importar librerías necesarias
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [13]:
import urllib.request

# URL directa al archivo dentro de la carpeta data/
url = "https://raw.githubusercontent.com/ebuitrago/demo-reg-clas-canciones/main/data/canciones.csv"

# Descargar y guardar en la carpeta actual
urllib.request.urlretrieve(url, "canciones.csv")
print("¡Archivo canciones.csv descargado con éxito!")

¡Archivo canciones.csv descargado con éxito!


In [14]:
# Cargar el archivo recién descargado
df = pd.read_csv("canciones.csv")

# Eliminamos explícitamente las variables no disponibles antes del lanzamiento
columnas_fuga = ["reproducciones_sem1", "es_hit"]
df_clean = df.drop(columns=columnas_fuga)

print(f"Total de registros: {len(df_clean)}")
print("Columnas conservadas para el modelo:", list(df_clean.columns))

Total de registros: 2000
Columnas conservadas para el modelo: ['bailabilidad', 'energia', 'valencia', 'acustica', 'tempo', 'duracion_min', 'volumen_db', 'anio', 'colaboracion', 'genero', 'popularidad']


In [15]:
# Feature derivada: duración centrada en 3.5 minutos al cuadrado
df_clean["duracion_c2"] = (df_clean["duracion_min"] - 3.5) ** 2

# Partición temporal imitando el escenario real
train_mask = df_clean["anio"] <= 2019
test_mask = df_clean["anio"] >= 2020

df_train = df_clean[train_mask]
df_test = df_clean[test_mask]

# Separar variables explicativas (X) de la variable objetivo (y = popularidad)
X_train = df_train.drop(columns=["popularidad"])
y_train = df_train["popularidad"]

X_test = df_test.drop(columns=["popularidad"])
y_test = df_test["popularidad"]

print(f"Canciones en Entrenamiento (1995-2019): {len(X_train)}")
print(f"Canciones en Prueba (2020-2025): {len(X_test)}")

Canciones en Entrenamiento (1995-2019): 1585
Canciones en Prueba (2020-2025): 415


In [16]:
# Definición de variables numéricas y categóricas
col_numericas = ["bailabilidad", "energia", "valencia", "acustica", "tempo", "volumen_db", "duracion_min", "duracion_c2", "colaboracion", "anio"]
col_categoricas = ["genero"]

# Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), col_numericas),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), col_categoricas)
    ]
)

# Pipeline de scikit-learn
pipeline_modelo = Pipeline([
    ("preprocesamiento", preprocessor),
    ("modelo", Ridge(alpha=1.0))
])

# Entrenamiento con datos de entrenamiento
pipeline_modelo.fit(X_train, y_train)

# Predicción en conjunto de prueba
y_pred = pipeline_modelo.predict(X_test)

# Métricas de evaluación
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- RESULTADOS SOBRE PRUEBA (2020-2025) ---")
print(f"MAE  (Error absoluto medio): {mae:.2f} puntos")
print(f"RMSE (Error cuadrático medio): {rmse:.2f} puntos")
print(f"R²   (Varianza explicada):   {r2:.2f}")

--- RESULTADOS SOBRE PRUEBA (2020-2025) ---
MAE  (Error absoluto medio): 5.76 puntos
RMSE (Error cuadrático medio): 7.15 puntos
R²   (Varianza explicada):   0.72


In [17]:
# Exportar el modelo dentro de la carpeta actual
joblib.dump(pipeline_modelo, "modelo_popularidad.pkl")
print("¡Modelo guardado exitosamente como 'modelo_popularidad.pkl'!")

¡Modelo guardado exitosamente como 'modelo_popularidad.pkl'!
